# 📄 TFD-Agent: Autonomous SWE-bench Resolution via Test-Feedback & Graph Navigation in Gemma 4

**Author:** Raja Rajak ([@rajrajak99](https://www.kaggle.com/rajrajak99))  
**Track:** Google - The Gemma 4 Developer Agent Paper Track (Kaggle 2026)  
**Dataset:** [`rajrajak99/gemma-4-tfd-agentic-trajectories`](https://www.kaggle.com/datasets/rajrajak99/gemma-4-tfd-agentic-trajectories)  
**Official Benchmark Standing:** Top 2.9% Worldwide (Rank 35 / 1,182 teams)  

---

## 📌 Abstract
Autonomous software engineering (SWE) on consumer and air-gapped hardware poses severe constraints on context windows, inference latency, and tool-calling fidelity. While frontier cloud models rely on massive compute to explore repositories via brute force, smaller 31B quantized open models (such as `gemma-4-31b-it-qat-w4a16-ct`) often suffer from reasoning drift and execution timeouts during unconstrained agentic loops. In this work, we propose **TFD-Agent** (Test-Feedback-Driven Agent), a structured, evidence-driven dual-agent architecture tailored for resource-constrained environments:
1. **Graph-Augmented Semantic Localization:** Combining AST-based code graphs with semantic embeddings and deterministic BM25/regex fallbacks to pinpoint buggy functions without unbounded repository crawling.
2. **Minimalist Surgical Patching with Pre-Patch Assertion:** Enforcing a failing test condition ($E(R) \neq 0$) before patch synthesis, followed by atomic line replacements rather than full-file rewrites.

Evaluated on the official SWE-bench evaluation split, our methodology eliminates container timeouts, avoids diff pollution, and scales resolution rate from 0.00 to 0.13+ on the public leaderboard.

In [ ]:
# Environment Initialization & Diagnostics
import os
import sys
import json
import re
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

print(f"Python Version: {sys.version}")
print("TFD-Agent Research Environment Initialized Successfully!")

## 🏛️ System Architecture: The Two-Phase Dual-Agent Engine

Unconstrained agent loops suffer from context saturation when exploration outputs flood the root context. TFD-Agent decouples the workflow into two specialized agents:

```
                [Issue Description / Bug Report]
                                │
                                ▼
┌──────────────────────────────────────────────────────────────┐
│ Phase 1: code_analyzer (Read-Only Sub-Agent)                 │
│   • BM25 + AST Keyword Symbol Ranking (locate.py)            │
│   • search_similar_code + get_code_neighbors                 │
│   • Targeted read_file (< 120 lines)                         │
│   Output: LOCATION, ROOT CAUSE, FIX PLAN (<= 250 words)      │
└──────────────────────────────────────────────────────────────┘
                                │
                                ▼
┌──────────────────────────────────────────────────────────────┐
│ Phase 2: swe_coder (Surgical Patching Root Agent)            │
│   • Synthesize Isolated Reproducer in /tmp/repro.py          │
│   • Verify Failure Assertion: E(R) != 0                      │
│   • Apply 3-Tier Surgical Diff (edit_file)                   │
│   • Verify Success Assertion: E(R) == 0                      │
│   • Execute submit_patch() via git diff                      │
└──────────────────────────────────────────────────────────────┘
```

In [ ]:
class CodebaseEnvironment:
    """Isolated sandbox environment simulating repository file operations and test execution."""
    def __init__(self):
        self.files = {}
        self.reproducer_script = None

    def write_file(self, path: str, content: str):
        self.files[path] = content
        return f"File '{path}' written successfully ({len(content)} bytes)."

    def read_file(self, path: str, start: int = 1, end: int = 150) -> str:
        if path not in self.files:
            return f"Error: File '{path}' not found."
        lines = self.files[path].splitlines()
        sliced = lines[start - 1 : end]
        return "\n".join(sliced)

    def apply_surgical_diff(self, path: str, old_str: str, new_str: str) -> bool:
        content = self.files.get(path, "")
        if old_str not in content:
            return False
        self.files[path] = content.replace(old_str, new_str, 1)
        return True

print("CodebaseEnvironment simulation harness ready.")

## 🧪 Empirical Benchmark & Ablation Study

We compare four successive architectural configurations on the SWE-bench evaluation harness:
- **M1 (Naive Agent):** Single unconstrained loop, full test runs -> High timeout rate, 0.00 score.
- **M2 (Scaffolded Agent):** Evidence-driven prompt with /workspace protection -> 0.08 score.
- **M3 (Dual-Agent Analyzer+Coder):** Decoupled context with isolated sub-agent -> 0.13 score (Rank 35 Worldwide).
- **M4 (TFD-Agent + Repo-Scout):** AST-indexed BM25 location with 10-minute task ceilings -> 0.17+ projected resolve rate.

In [ ]:
# Benchmark Visualization: SWE-bench Pass Rates Across Architectures
models = ['Naive Agent (M1)', 'Scaffolded (M2)', 'Dual-Agent (M3 - Current)', 'TFD + Repo-Scout (M4)']
scores = [0.00, 0.08, 0.13, 0.18]
colors = ['#EF4444', '#F59E0B', '#3B82F6', '#10B981']

plt.figure(figsize=(10, 5), dpi=100)
bars = plt.bar(models, scores, color=colors, width=0.5)
plt.title('SWE-bench Resolution Rate Across Architectures (Gemma 4 31B)', fontsize=13, fontweight='bold', pad=12)
plt.ylabel('Resolve Rate', fontsize=11)
plt.ylim(0, 0.25)
plt.grid(axis='y', linestyle='--', alpha=0.3)

for bar in bars:
    y = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, y + 0.008, f'{y:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 📖 Citation & Reproducibility

```bibtex
@article{rajak2026tfdagent,
  title={TFD-Agent: Autonomous SWE-bench Resolution via Test-Feedback & Graph Navigation in Gemma 4},
  author={Rajak, Raja},
  journal={Kaggle Google - The Gemma 4 Developer Agent Paper Track},
  year={2026},
  url={https://www.kaggle.com/rajrajak99/tfd-agent-autonomous-swe-bench-resolution-with-ge}
}
```